In [ ]:
import warnings
warnings.filterwarnings('ignore')
import janitor
janitor
import os
import pandas as pd
import numpy as np
import string
# import unidecode
from tqdm import tqdm
from datetime import datetime

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)

FP_PARSED_GEOLOC = "../3-parseGeolocation/output/parsed-geoloc.csv"
FP_RETURNS2018 = "../5-election2018Data/output/counties-election-results.csv"
FP_POLITICIAN_ETHNIC = "../5-election2018Data/output/politicians-with-raceindicators.csv"
FP_COUNTY_CENSUS = "./election-context-2018.csv"
FP_PRESIDENTIAL_RETURNS = "../9-election2016Data/input/countypres_2000-2016.csv"
FP_ACS1519 = "../6-ACS_2015_19/output/acs_2015_2019.csv"
FP_BOTSCORE = "../4-classifyAccounts/out/bot-classification.csv"
FP_RETURNS2014 = "input/dlatlast/2014_4_0_2/CSV/COUNTY/2014_4_0_2.csv"
mappingBinaries = {'yes': 1, 'no': 0}

## Merging geolocation and tweets

In [ ]:
# Get human/bot classification
classified_human_accounts = (
    pd.read_csv(FP_BOTSCORE)
    .fillna(0)
    .query("bot_score<=.5")
    ["user_id"]
    .to_list()
)
classified_human_accounts[:5]

In [ ]:
df = (
    pd.read_csv(FP_PARSED_GEOLOC)
    .clean_names()
    .dropna(subset=["primary_county"])
    # Drop classified bot accounts
    .query("author_id in @classified_human_accounts")
)
print(len(df))
df.head()

**Load in tweets data**
* input files from ../1-downloadTweets/tweets-data/*month*2018.csv

In [ ]:
tweetsFilesList = ['jan2018.csv',
                   'feb2018.csv',
                   'mar2018.csv',                   
                   'apr2018.csv',
                   'may2018.csv', 
                   'jun2018.csv',                   
                   'jul2018.csv',
                   'aug2018.csv',
                   'sep2018.csv',
                   'oct2018.csv',
                   'nov2018.csv']

# get all tweets in 2018 (Jan 2018 to 6 Nov 2018)
dfTweets = pd.DataFrame()

for fileName in tqdm(tweetsFilesList):
    
    _df = pd.read_csv(os.path.join('../1-downloadTweets', 'tweets-data', fileName), 
                      index_col=0)
    dfTweets = pd.concat([dfTweets, _df], ignore_index=True)

print(len(dfTweets))
dfTweets.head()

In [ ]:
df_merged_tweets_users = (
    df
    .remove_columns(["username"])
    .drop_duplicates(["author_id"])
    # Merge user to tweets
    .merge(dfTweets, on='author_id', how="inner", validate="1:m")
    # ====================================================
    .assign(
        tweet_length=lambda df_: df_['content'].str.len(),
        date=lambda df_: pd.to_datetime(df_['date']),
    )
    # ====================================================
    # Clean up county names
    .assign(
        **{
            'primary_county': lambda df_: (df_['primary_county']
                .str.replace('(city)', '', regex=False)
                .str.replace('city', '', regex=False)
                .str.replace('(', '', regex=False)
                .str.replace(')', '', regex=False)
                .str.strip()
            )
        }
    )
)
df_merged_tweets_users

In [ ]:
df_county_tweets = (
    df_merged_tweets_users
    .groupby(['primary_county', 'state_short', 'state_full'])
    .size()
    .to_frame('tweet_count')
    .reset_index()
    .remove_columns("state_full")
    .rename(columns={'primary_county': 'county', 'state_short': 'state'})
)
df_county_tweets

## Merging with 2018 election results

In [ ]:
# read in 2018 election results
df_election = (
    pd.read_csv(FP_RETURNS2018)
    .clean_names()
    # ==================================================
    # Clean names
    .pipe(lambda df_: df_.rename(columns=dict(zip(df_.columns, 
                                                  df_.columns.tolist()[:1] + ['state_name'] + df_.columns.tolist()[2:]))))
    .assign(
        county=lambda df_: (df_['county']
            .str.lower()
            .str.replace('(city)', '', regex=False)
            .str.replace('(', '', regex=False)
            .str.replace(')', '', regex=False)
            .str.strip()
        ),
        state_name=lambda df_: df_['state_name'].str.lower()
    )
)
df_election

In [ ]:
df_merge = (
    df_election
    .merge(df_county_tweets, how='left', on=['state', 'county'])
    .assign(tweet_count=lambda df_: df_["tweet_count"].fillna(0))
)
df_merge

## Merge with 2018 politician ethnic data

In [ ]:
columnsToKeep = ['politician', 'party', 'incumbent', 'gender', 'black', 'race difference', 'np1-white', 
                 'np2-white', 'np1-black', 'np2-black', 'np1-hispanic', 'np2-hispanic', 'np1-others', 'np2-others']

df_politician = (
    pd.read_csv(FP_POLITICIAN_ETHNIC, usecols=columnsToKeep)
    .clean_names()
    .assign(
        woman=lambda df_: np.where(df_['gender'] == 'male', 0, 1),
        republican=lambda df_: np.where(df_['party'] == 'rep', 1, 0),
        democrat=lambda df_: np.where(df_['party'] == 'dem', 1, 0),
        independent=lambda df_: np.where((df_['party'] != 'dem') & (df_['party'] != 'rep'), 1, 0)
    )
    
)
df_politician

In [ ]:
df_merge = (
    df_merge
    .merge(df_politician, how="left", on="politician", validate="m:1")
    # ====================================================
    .assign(**{
        col: lambda df_, c=col: df_[c].map(mappingBinaries) 
        for col in ['open_seat', 'no_main_challenger', 'multiple_rep', 'multiple_dem', 
                   'third_party', 'all_women', 'all_men', 'same_gender', 
                   'woman_challenger', 'black', 'incumbent']
    })    
)
assert (df_merge["republican"] + df_merge["democrat"] + df_merge["independent"] == 1).all()
df_merge

## Merge with county-level demographics

### Correcting State names

In [ ]:
# correcting state name (e.g. from rhodeisland to rhode island)
correctStateNames = {'newhampshire': 'new hampshire', 
                     'newjersey': 'new jersey', 
                     'newmexico': 'new mexico', 
                     'northcarolina': 'north carolina', 
                     'northdakota': 'north dakota', 
                     'rhodeisland': 'rhode island', 
                     'southcarolina': 'south carolina', 
                     'southdakota': 'south dakota', 
                     'westvirginia': 'west virginia', 
                     'delare': 'delaware', 
                     'newyork': 'new york'}

df_merge['state_name'] = df_merge['state_name'].apply(lambda x: correctStateNames[x] if x in correctStateNames else x)

### Correcting County names

In [ ]:
# correcting county names before merging with county demographics data
correctNewJerseyCounties = {'glouchester': 'camden'}

correctVirginiaCounties = {'charles': 'charles city', 
                           'glouchester': 'gloucester', 
                           'james': 'james city', 
                           'king & queen': 'king and queen'}


for ix in range(0, len(df_merge)):        
    if df_merge.at[ix, 'state_name'] == 'new jersey':
        try:
            df_merge.at[ix, 'county'] = correctNewJerseyCounties[df_merge.at[ix, 'county']]
        except KeyError:
            pass

    if df_merge.at[ix, 'state_name'] == 'virginia':
        try:
            df_merge.at[ix, 'county'] = correctVirginiaCounties[df_merge.at[ix, 'county']]
        except KeyError:
            pass


for ix in range(0, len(df_merge)):
    if df_merge.at[ix, 'state_name'] == 'indiana':
        if df_merge.at[ix, 'county'] == 'newtown':
            df_merge.at[ix, 'county'] = 'fountain'

### Merge

In [ ]:
df_county_demo = (
    pd.read_csv(FP_COUNTY_CENSUS)
    .assign(
        state=lambda df_: df_['state'].str.lower().str.strip(),
        county=lambda df_: df_['county'].str.lower().str.strip()
    )
    .drop_duplicates(subset=['state', 'county'], keep='first')
    .rename_column("state", "state_name")
)
df_county_demo

In [ ]:
df_merge = df_merge.merge(df_county_demo, how="left", on=['state_name', 'county'])
df_merge.head()

## Get 2008 Presidential data

In [ ]:
df_presidential_returns = (
    pd.read_csv(FP_PRESIDENTIAL_RETURNS)
    .query('year >= 2008')
    .dropna(subset=["FIPS", "candidatevotes"])
    .assign(
        FIPS=lambda df_: df_['FIPS'].astype('int'),
        candidatevotes=lambda df_: df_['candidatevotes'].astype('int')
    )
    .remove_columns(['totalvotes', 'office', 'version', 'candidate', 'state_po'])
)
df_presidential_returns

In [ ]:
df_pres_dem2008 = (
    df_presidential_returns
    .query('year == 2008 & party == "democrat"')
    .filter(['FIPS', 'candidatevotes'])
    .rename_column("FIPS", "fips")
    .rename_column("candidatevotes", "demPresVotes2008")
)

df_pres_rep2008 = (
    df_presidential_returns
    .query('year == 2008 & party == "republican"')
    .filter(['FIPS', 'candidatevotes'])
    .rename_column("FIPS", "fips")
    .rename_column("candidatevotes", "repPresVotes2008")
)

df_pres_other2008 = (
    df_presidential_returns
    .query('year == 2008 & party != "republican" & party != "democrat"')
    .filter(['FIPS', 'candidatevotes'])
    .rename_column("FIPS", "fips")
    .rename_column("candidatevotes", "otherPresVotes2008")
)
df_pres_dem2008

In [ ]:
_MERGE_OPTS = dict(how="left", on="fips", validate="m:1")
df_merge = (
    df_merge
    .merge(df_pres_dem2008, **_MERGE_OPTS)
    .merge(df_pres_rep2008, **_MERGE_OPTS)
    .merge(df_pres_other2008, **_MERGE_OPTS)
)
df_merge

In [ ]:
## Fix the error for Doug Lamborn and Douglas Randall
df_merge.loc[df_merge['politician'] == 'Doug Lamborn', 'incumbent'] = 1
df_merge.loc[df_merge['politician'] == 'Douglas Randall', 'incumbent'] = 0

## Merge with ACS2015-19

In [ ]:
df_merge = (
    df_merge
    .merge(pd.read_csv(FP_ACS1519), how="left", on="fips", validate="m:1")
)
df_merge

## Generate aggregates

In [ ]:
df_merge.info()

### Total 2018 House county votes

In [ ]:
_df = (
    df_merge
    .filter(['county', 'state', 'district', 'republican', 'democrat', 'independent', 'votes'])
    .groupby(by=['state', 'county'])
    [['votes']]
    .sum().reset_index()
    .rename_column("votes", "district_county_votes")
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
_df

In [ ]:
_df = (
    df_merge
    .filter(['county', 'state', 'district', 'republican', 'democrat', 'votes'])
    .query("republican==1 | democrat == 1")
    .groupby(by=['state', 'county'])
    [['votes']]
    .sum().reset_index()
    .rename_column("votes", "twoparty_district_county_votes")
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
assert (df_merge["twoparty_district_county_votes"] <= df_merge["district_county_votes"]).all()
_df

### District Incumbency

In [ ]:
_df = (
    df_merge
    .filter(['district', 'incumbent', 'woman'])
    .assign(placeholder=lambda df_: df_['incumbent'] * df_['woman'])
    .groupby(by=['district'])[['placeholder']]
    .sum().reset_index()
    .assign(womanincumbent=lambda df_: np.where(df_['placeholder'] > 0, 1, 0 ))
    .remove_columns(["placeholder"])
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'womanincumbent': 0})
)
_df

In [ ]:
_df = (
    df_merge
    .filter(['district', 'incumbent', 'woman'])
    .drop_duplicates("district", ignore_index=True)
    .assign(
        man=lambda df_: 1 - df_['woman'],
        manincumbent=lambda df_: df_['incumbent'] * df_['man']
    )
    .remove_columns(["incumbent", "woman", "man"])
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'manincumbent': 0})
)
_df

### 2016 County Presidential election votes

In [ ]:
_df = (
    df_merge
    .filter(['state', 'county', 'trump16', 'clinton16', 'otherpres16'])
    .drop_duplicates(ignore_index=True)
    .assign(totalpresvotes2016=lambda df_: df_[['trump16', 'clinton16', 'otherpres16']].sum(axis=1))
    .filter(["state", "county", "totalpresvotes2016"])
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
_df

In [ ]:
_df = (
    df_merge
    .filter(['state', 'county', 'trump16', 'clinton16'])
    .drop_duplicates(ignore_index=True)
    .assign(totaltwoptypresvotes2016=lambda df_: df_[['trump16', 'clinton16']].sum(axis=1))
    .filter(["state", "county", "totaltwoptypresvotes2016"])
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
assert (df_merge["totaltwoptypresvotes2016"] <= df_merge["totalpresvotes2016"]).all()

_df

### 2012 County Presidential election votes

In [ ]:
_df = (
    df_merge
    .filter(['state', 'county', 'romney12', 'obama12', 'otherpres12'])
    .drop_duplicates(ignore_index=True)
    .assign(totalpresvotes2012=lambda df_: df_[['romney12', 'obama12', 'otherpres12']].sum(axis=1))
    .filter(["state", "county", "totalpresvotes2012"])
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
_df

In [ ]:
_df = (
    df_merge
    .filter(['state', 'county', 'romney12', 'obama12'])
    .drop_duplicates(ignore_index=True)
    .assign(totaltwoptypresvotes2012=lambda df_: df_[['romney12', 'obama12']].sum(axis=1))
    .filter(["state", "county", "totaltwoptypresvotes2012"])
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
assert (df_merge["totaltwoptypresvotes2012"] <= df_merge["totalpresvotes2012"]).all()
_df

### Total 2016 House county votes

In [ ]:
_df = (
    df_merge
    .filter(['county', 'state', 'district', 'demhouse16',  'rephouse16',  'otherhouse16'])
    .drop_duplicates(["county", "state"], keep='first', ignore_index=True)
    .assign(totalhousevotes2016=lambda df_: df_[['demhouse16',  'rephouse16',  'otherhouse16']].sum(axis=1))
    .filter(["state", "county", "totalhousevotes2016"])
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
_df

In [ ]:
_df = (
    df_merge
    .filter(['county', 'state', 'district', 'demhouse16',  'rephouse16',  'otherhouse16'])
    .drop_duplicates(["county", "state"], keep='first', ignore_index=True)
    .assign(totaltwoptyhousevotes2016=lambda df_: df_[['demhouse16',  'rephouse16']].sum(axis=1))
    .filter(["state", "county", "totaltwoptyhousevotes2016"])
)
df_merge = df_merge.merge(_df, how="left", on=["state", "county"], validate="m:1")
assert (df_merge["totaltwoptyhousevotes2016"] <= df_merge["totalhousevotes2016"]).all()
_df

### At least one woman challenger

In [ ]:
_df = (
    df_merge
    .filter(['district', 'woman', 'incumbent'])
    .query("incumbent==0")
    .remove_columns("incumbent")
    .groupby("district")
    .sum()
    .reset_index()
    .assign(atLeast1WomanChallenger=lambda df_: np.where(df_['woman'] > 0, 1, 0 ))
    .remove_columns("woman")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'atLeast1WomanChallenger': 0})
)
_df

In [ ]:
_df = (
    df_merge
    .filter(['district', 'woman', 'democrat', 'incumbent'])
    .query("incumbent==0")
    .query('democrat==1')
    .remove_columns(["incumbent", "democrat"])
    .groupby("district")
    .sum()
    .reset_index()
    .assign(atLeast1DemWomanChallenger=lambda df_: np.where(df_['woman'] > 0, 1, 0 ))
    .remove_columns("woman")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'atLeast1DemWomanChallenger': 0})
)
assert (df_merge["atLeast1DemWomanChallenger"] <= df_merge["atLeast1WomanChallenger"]).all()
_df

In [ ]:
_df = (
    df_merge
    .filter(['district', 'woman', 'republican', 'incumbent'])
    .query("incumbent==0")
    .query('republican==1')
    .remove_columns(["incumbent", "republican"])
    .groupby("district")
    .sum()
    .reset_index()
    .assign(atLeast1RepWomanChallenger=lambda df_: np.where(df_['woman'] > 0, 1, 0 ))
    .remove_columns("woman")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'atLeast1RepWomanChallenger': 0})
)
assert (df_merge["atLeast1RepWomanChallenger"] <= df_merge["atLeast1WomanChallenger"]).all()
_df

### Open seat

In [ ]:
_df = (
    df_merge
    .filter(['district', 'incumbent'])
    .groupby('district')['incumbent']
    .sum()
    .reset_index()
    .assign(openSeat=lambda df_: (df_['incumbent'] == 0).astype(int))
    .remove_columns("incumbent")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'openSeat': 0})
)
_df

### Woman incumbent

In [ ]:
_df = (
    df_merge
    .filter(['district', 'incumbent', 'woman'])
    .query("incumbent==1")
    .groupby('district')['woman']
    .sum()
    .reset_index()
    .assign(womanIncumbent=lambda df_: (df_['woman'] > 0).astype(int))
    .remove_columns("woman")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'womanIncumbent': 0})
)
_df

### Man incumbent

In [ ]:
_df = (
    df_merge
    .filter(['district', 'incumbent', 'woman'])
    .query("incumbent==1")
    .groupby('district')['woman']
    .sum()
    .reset_index()
    .assign(manIncumbent=lambda df_: (df_['woman'] == 0).astype(int))
    .remove_columns("woman")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'manIncumbent': 0})
)
assert (df_merge["manIncumbent"] + df_merge["womanIncumbent"]+df_merge["openSeat"]==1).all()
_df

### Rep incumbent

In [ ]:
_df = (
    df_merge
    .filter(['district', 'incumbent', 'republican'])
    .query("incumbent==1")
    .groupby('district')['republican']
    .sum()
    .reset_index()
    .assign(repIncumbent=lambda df_: (df_['republican'] > 0).astype(int))
    .remove_columns("republican")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'repIncumbent': 0})
)
_df

### Dem incumbent

In [ ]:
_df = (
    df_merge
    .filter(['district', 'incumbent', 'democrat'])  # or whatever your Democrat column is called
    .query("incumbent==1")
    .groupby('district')['democrat']
    .sum()
    .reset_index()
    .assign(demIncumbent=lambda df_: (df_['democrat'] > 0).astype(int))
    .remove_columns("democrat")
)
df_merge = (
    df_merge
    .merge(_df, how="left", on="district", validate="m:1")
    .fillna({'demIncumbent': 0})
)
assert (df_merge["demIncumbent"] + df_merge["repIncumbent"]+df_merge["openSeat"]<=1).all()

_df

### Dem house votes 2018

In [ ]:
_df = (
    df_merge
    .filter(['state', 'county', 'democrat', 'votes'])
    .query("democrat==1")
    .groupby(['state', 'county'])['votes']
    .sum()
    .reset_index()
    .rename(columns={'votes': 'demhouse18'})
)
df_merge = (
    df_merge
    .merge(_df, on=['state', 'county'], how='left')
    .fillna({'demhouse18': 0})
)
_df

In [ ]:
_df = (
    df_merge
    .filter(['state', 'county', 'republican', 'votes'])
    .query("republican==1")
    .groupby(['state', 'county'])['votes']
    .sum()
    .reset_index()
    .rename(columns={'votes': 'rephouse18'})
)
df_merge = (
    df_merge
    .merge(_df, on=['state', 'county'], how='left')
    .fillna({'rephouse18': 0})
)
_df

In [ ]:
_df = (
    df_merge
    .filter(['state', 'county', 'independent', 'votes'])
    .query("independent==1")
    .groupby(['state', 'county'])['votes']
    .sum()
    .reset_index()
    .rename(columns={'votes': 'otherhouse18'})
)
df_merge = (
    df_merge
    .merge(_df, on=['state', 'county'], how='left')
    .fillna({'otherhouse18': 0})
)
_df

## 2014 turnout

In [ ]:
# Add 2014 turnout
df_merge = (
    df_merge
    .merge(
        (
            pd.read_csv(FP_RETURNS2014, skiprows=1)
            .filter(["fips", "totalvote"])
            .rename_column("totalvote", "totalHouse14")
        ), how="left", on="fips", validate="m:1"
    )
)

## Export

In [ ]:
df_merge.to_stata("./output/panel.dta", write_index=False, version=117)
df_merge.to_csv("./output/panel.csv", index=False)
display(df_merge)
df_merge.info(verbose=True, show_counts=True, memory_usage=True)